# Request shapes and request boundaries — a decision report

**Status:** open. Two questions for the author: **Q28** (which rule labels a
turn's shape) and **Q29** (how a participant's turns are grouped into
requests). This notebook records how both were approached, and the evidence
gathered before either is ruled on. Every number below is computed from run
`42cab17a` (full suite, 29 cases, `n = 1`, 2026-09-23). No new run was made,
and nothing was written to the database.

**In short**

1. A seven-shape rule (`shapes7`) lands every case on a stable label. The
   metamorphic relations do not disturb it, and it produces no `other`.
2. No rule based on the trace can produce `advisory` on this suite: the agent
   consults a reference tool before it gives advice.
3. The participant round records turns but not requests. A rule that splits a
   run wherever the shape changes recovers **83.5 %** of request boundaries in
   runs built by chaining recorded executions. Every miss is two
   same-shape requests merging, and no merged request gets a wrong label.
4. The opposite error, one request split in two, cannot be measured with the
   recorded cases.

## 1. How this came up

The participant round labels every agent turn automatically, from the tools it
called. Two labels existed before this report:

| label | defined in | grain | problem for the participant analysis |
|---|---|---|---|
| `request_type` | D09 | 4 classes, one per function of §1.4 | too coarse to say *where* uncovered participant turns concentrate (§4.3 (3)) |
| coverage signature | D24 | exact tool sequence + buckets | too fine to carry a failure rate per class |

A **request shape** would sit between the two. The author's starting
hypothesis was to take the shapes clearly identifiable in the 29 cases:
enough classes to be informative, few enough to keep cells populated. The
test was to check that hypothesis against a full-suite `n = 1` run before
committing to it. The most recent verification run, `42cab17a`, is used
rather than a new one.

A shape is derived after the fact from `tool_call_trace` (`src/shapes.py`). No
application change and no event-model change are involved.

## 2. The candidate rules

Each rule is a pure function of the tool names one turn called. Reading the
trip (`get_trip_details`, `list_trip_blocks`, `get_itinerary`) is context and
ignored by every rule.

| rule | classes |
|---|---|
| `d09` | D09's precedence, re-derived: scheduling › venue_lookup › factual_qa › advisory |
| `shapes7` | advisory · factual.reference · factual.climate · factual.forecast · venue.search · scheduling.draft · scheduling.edit · **other** |
| `shapes9` | `shapes7` with scheduling.edit split into move / place / remove |
| `families` | the set of D09 families present, joined (Q08's option b) |

`shapes7` and `shapes9` take **no precedence**. A turn that does two things is
`other`, because a precedence would silently decide which half counts, and
the mixed turn is exactly what participants may do and the suite does not.
The one exception is a search that feeds a draft: that turn is the draft,
because it is the suite's own grounded-plan shape.

Three checks decide whether a rule is trusted on participants:

1. **Label vs. declaration.** Each execution lands on the label its case's
   declared tools imply.
2. **Metamorphic consistency.** A follow-up keeps its seed's label, unless
   its relation changes which tool is needed.
3. **Sizes and `other`.** No class is empty, and `other` is near zero on the
   suite.

In [1]:
import sys, pathlib, random, itertools, collections
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import pandas as pd

from src import survey, shapes
from src.cases import load_cases
from src.coverage import tools_required_by, tools_alternatively_required_by

pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 60)

# Pinned: the stored outputs describe this run and no other.
RUN_ID = "42cab17a-4903-4e39-b00c-ace76b7929e4"
RULES = list(shapes.RULES)

print(survey.headline(RUN_ID))
print()
print(f"shape rule version {shapes.REQUEST_SHAPE_RULE_VERSION}; suite holds {len(load_cases())} cases")

run 42cab17a  started 2026-09-23T17:52:07
  notes         FULL SUITE n=1 post-D74..D80 - verification
  agent         gemini-3.5-flash (released 2026-05-19)
  api           0.1.3 @ 5af96abf452b
  suite         031589fd9658
  executions    29  (+0 superseded)
  failed        0
  scored        29, mean 9.34
  cost          $0.6498



shape rule version 0-candidates; suite holds 29 cases


## 3. Every turn, under every rule

One row per agent turn. `request_type` is the label the app stored (A03). The
`d09` column re-derives the same rule here, and the two must agree. Otherwise
the comparison below would be between two codebases, not two rules.

In [2]:
turns = survey.turn_frame(RUN_ID)
for rule in RULES:
    turns[rule] = turns["tool_names"].map(shapes.RULES[rule])

disagree = turns[turns["d09"] != turns["request_type"]]
print(f"{len(turns)} turns; d09 re-derived vs stored request_type: {len(disagree)} disagreements")
turns[["case", "turn", "tool_names", "request_type", "shapes7", "shapes9", "families"]]

35 turns; d09 re-derived vs stored request_type: 0 disagreements


,case,turn,tool_names,request_type,shapes7,shapes9,families
0,advisory_packing_list,1,[get_region_climate],factual_qa,factual.climate,factual.climate,factual_qa
1,advisory_packing_list,2,[],advisory,advisory,advisory,advisory
2,advisory_packing_list__inv1_paraphrase,1,[get_region_climate],factual_qa,factual.climate,factual.climate,factual_qa
3,advisory_packing_list__inv1_paraphrase,2,[],advisory,advisory,advisory,advisory
4,advisory_packing_list__inv3_irrelevant_detail,1,[get_region_climate],factual_qa,factual.climate,factual.climate,factual_qa
5,advisory_packing_list__inv3_irrelevant_detail,2,[],advisory,advisory,advisory,advisory
6,advisory_sintra_full_or_half_day,1,[read_wikipedia],factual_qa,factual.reference,factual.reference,factual_qa
7,advisory_sintra_full_or_half_day__inv1_paraphrase,1,[read_wikipedia],factual_qa,factual.reference,factual.reference,factual_qa
8,advisory_sintra_full_or_half_day__inv4_german,1,[read_wikipedia],factual_qa,factual.reference,factual.reference,factual_qa
9,advisory_sintra_full_or_half_day__inv4_german,2,[],advisory,advisory,advisory,advisory


## 4. Check 1 — label vs. declaration

Per execution, the interaction's label is compared with the label the case's
declared tools imply (`tools_required_by`, plus one member per `any_of`
group). The interaction's label ignores a closing turn that called nothing
(`shapes.interaction_label`).

In [3]:
cases = {c.case_id: c for c in load_cases()}

rows = []
for case_id, frame in turns.groupby("case"):
    case = cases[case_id]
    row = {"case": case_id, "function": case.function, "relation": case.metamorphic_relation or "seed"}
    for rule in RULES:
        declared = shapes.declared_label(tools_required_by(case), tools_alternatively_required_by(case), rule)
        observed = shapes.interaction_label(list(frame[rule]))
        row[rule] = observed
        row[f"{rule} ok"] = observed == declared
        row[f"{rule} declared"] = declared
    rows.append(row)
execs = pd.DataFrame(rows)

print(f"{len(execs)} executions")
pd.DataFrame({rule: {"agree": int(execs[f"{rule} ok"].sum()), "of": len(execs)} for rule in RULES}).T

29 executions


,agree,of
d09,23,29
shapes7,23,29
shapes9,23,29
families,23,29


In [4]:
# Where the observed label and the declaration part ways, under any rule.
cols = ["case"] + [c for rule in RULES for c in (f"{rule} declared", rule)]
execs.loc[~execs[[f"{r} ok" for r in RULES]].all(axis=1), cols]

,case,d09 declared,d09,shapes7 declared,shapes7,shapes9 declared,shapes9,families declared,families
0,advisory_packing_list,advisory,factual_qa,advisory,factual.climate,advisory,factual.climate,advisory,factual_qa
1,advisory_packing_list__inv1_paraphrase,advisory,factual_qa,advisory,factual.climate,advisory,factual.climate,advisory,factual_qa
2,advisory_packing_list__inv3_irrelevant_detail,advisory,factual_qa,advisory,factual.climate,advisory,factual.climate,advisory,factual_qa
3,advisory_sintra_full_or_half_day,advisory,factual_qa,advisory,factual.reference,advisory,factual.reference,advisory,factual_qa
4,advisory_sintra_full_or_half_day__inv1_paraphrase,advisory,factual_qa,advisory,factual.reference,advisory,factual.reference,advisory,factual_qa
5,advisory_sintra_full_or_half_day__inv4_german,advisory,factual_qa,advisory,factual.reference,advisory,factual.reference,advisory,factual_qa


**All four rules miss the same six executions, and they are the six advisory
cases.** For every advisory question, the agent consulted a reference tool
before it answered: `get_region_climate` for the packing list,
`read_wikipedia` for Sintra's full or half day. A rule that reads the trace
therefore cannot produce `advisory` on this suite.

This is not a defect in any rule. It is where what was *asked* (the case's
`function`) and what was *done* (the trace) diverge. It reaches beyond Q28:
P2's per-`request_type` failure rates will have no advisory row in either
round. In the participant round, an `advisory` label would mean a turn that
called no tool at all, a shape the suite never produced.

## 5. Check 2 — metamorphic consistency

Each follow-up is compared with its seed. The table shows a single label where
the relation kept it, and `seed → variant` where it did not.

In [5]:
label = execs.set_index("case")
meta = []
for case in cases.values():
    if not case.base_case_id or case.case_id not in label.index:
        continue
    row = {"variant": case.case_id.split("__", 1)[1], "seed": case.base_case_id}
    for rule in RULES:
        a, b = label.at[case.base_case_id, rule], label.at[case.case_id, rule]
        row[rule] = b if a == b else f"{a} → {b}"
    meta.append(row)
pd.DataFrame(meta).sort_values(["seed", "variant"]).reset_index(drop=True)

,variant,seed,d09,shapes7,shapes9,families
0,inv1_paraphrase,advisory_packing_list,factual_qa,factual.climate,factual.climate,factual_qa
1,inv3_irrelevant_detail,advisory_packing_list,factual_qa,factual.climate,factual.climate,factual_qa
2,inv1_paraphrase,advisory_sintra_full_or_half_day,factual_qa,factual.reference,factual.reference,factual_qa
3,inv4_german,advisory_sintra_full_or_half_day,factual_qa,factual.reference,factual.reference,factual_qa
4,dir1_the_earthquake,factual_qa_alfama_history,factual_qa,factual.reference,factual.reference,factual_qa
5,inv1_paraphrase,factual_qa_alfama_history,factual_qa,factual.reference,factual.reference,factual_qa
6,dir1_july_instead,factual_qa_lisbon_november_weather,factual_qa,factual.climate,factual.climate,factual_qa
7,inv1_paraphrase,factual_qa_lisbon_november_weather,factual_qa,factual.climate,factual.climate,factual_qa
8,dir2_on_the_9th,scheduling_belem_morning,scheduling,scheduling.draft,scheduling.draft,scheduling+venue_lookup
9,dir3_a_full_day,scheduling_belem_morning,scheduling,scheduling.draft,scheduling.draft,scheduling+venue_lookup


Every invariance relation keeps its label under every rule. Only `shapes9`
lets a directional relation show: `dir1_put_the_monastery_on` and
`dir4_off_the_itinerary` change the tool the request needs, and under
`shapes9` they change label. Under the other rules they are indistinguishable
from the seed.

## 6. Check 3 — class sizes and `other`

Executions per label. A class that a rule can produce but the suite never reaches is a class whose participant turns would all be uncovered.

In [6]:
producible = {
    "d09": ["scheduling", "venue_lookup", "factual_qa", "advisory"],
    "shapes7": ["advisory", "factual.reference", "factual.climate", "factual.forecast",
                "venue.search", "scheduling.draft", "scheduling.edit", "other"],
    "shapes9": ["advisory", "factual.reference", "factual.climate", "factual.forecast", "venue.search",
                "scheduling.draft", "scheduling.move", "scheduling.place", "scheduling.remove", "other"],
}
sizes = pd.DataFrame({rule: execs[rule].value_counts().reindex(classes, fill_value=0)
                      for rule, classes in producible.items()})
sizes.apply(lambda col: col.map(lambda v: "·" if pd.isna(v) else int(v)))

,d09,shapes7,shapes9
advisory,0,0,0
factual.climate,·,7,7
factual.forecast,·,2,2
factual.reference,·,6,6
factual_qa,15,·,·
other,·,0,0
scheduling,7,·,·
scheduling.draft,·,3,3
scheduling.edit,·,4,·
scheduling.move,·,·,2


`other` is empty: the suite contains no mixed turn at all. The one
combination it produces, search then draft, is absorbed by design. So any
participant turn that does two things is uncovered by construction, and the
share of such turns is part of the P1 finding.

`shapes7` reaches six of its seven named classes, with two to seven
executions each. `shapes9`'s place and remove classes hold one execution
each, too few to carry a rate even at the baseline's repetitions.

**Where Q28 stood after the three checks:** `shapes7`, with `other` reported
as its own row. Separately, `advisory` must either mean "called no tool" or
the suite needs a case that reaches it.

> **Later runs (D107, 2026-09-29).** This holds for `42cab17a` (29 cases)
> only. On the 31-case baseline `1a872c09` and regression run `e097f78e`,
> 17 and 13 requests are `other` (the D81 afternoon seed's draft + move,
> the packing list's weather + climate), and 19 and 20 *turns* are
> `advisory`, though no *request* is. See
> `07_shapes__baseline_1a872c09` (this analysis on the baseline),
> `08_segmentation__baseline_1a872c09` and `08_segmentation__regression_e097f78e`.


## 7. The second problem: the participant round has no request boundary

The checks above have a boundary for free: in the simulation, one case is one
request. The participant round does not:

- **Turns are separated.** Each agent turn is one `study_suggestions` row,
  with `run_id`, `turn_index`, `user_turn` and its trace (D07, D08).
- **Requests are not.** The only grouping above a turn is the Tonic run
  (`agent.runs.ts`). It collects a member's turns until they have been quiet
  for 30 minutes, so one run can hold a weather question, a venue search
  and a move.

This breaks three things. D24's signature needs the number of member turns in
an interaction. Labelled per turn, `advisory` would fill up with "thanks" and
"yes, do it". And the two rounds would count different units.

Options considered (Q29):

| | a request is… | cost |
|---|---|---|
| A | one turn (D07 as written) | follow-up and acknowledgement turns inflate `advisory` |
| B | one Tonic run | several requests in a run become one `other` |
| **C** | a run split wherever the acting shape changes; turns that call nothing join the request before them | two same-shape requests in a row merge; one request that changes shape splits |
| — | the agent tags a new request | **ruled out**: changes the agent's prompt and output (root rule 4), and is one more output that can be wrong |
| — | an offline segmenter model over `user_turn` | admissible, since it runs after the fact like the judge, but adds a model to pin and validate |

C is a pure, versioned rule over fields already logged (`shapes.segment`).
The question was whether it works.

## 8. The simulation: fusing recorded executions

To answer that without a new run, recorded executions are chained into
synthetic participant runs. In this run every execution is exactly one Tonic
run (checked below), so the original boundaries are known. They serve as the
ground truth that `segment` has to recover. The fusion happens in memory, and
the recorded rows are only read.

Two measurements:

- **All ordered pairs** of executions (29 × 28 = 812), exhaustively.
- **Random runs** of 3, 5 and 8 requests, 2 000 each, drawn with replacement
  (seed 28).

In [7]:
# Precondition: execution = Tonic run, so an execution is a known request.
from src import db
from src.execution import latest_attempts
from src.survey import _rows

executions, _ = latest_attempts(_rows("study_case_executions", "run_id", RUN_ID))
ids = [i for row in executions for i in (row.get("suggestion_ids") or [])]
run_of = {r["id"]: r["run_id"] for r in
          db.admin().table("study_suggestions").select("id, run_id").in_("id", ids).execute().data}
spanning = [row["case_id"] for row in executions
            if len({run_of[i] for i in row.get("suggestion_ids") or []}) != 1]
print(f"{len(executions)} executions, {len(set(run_of.values()))} Tonic runs, "
      f"executions spanning more than one run: {spanning or 'none'}")

29 executions, 29 Tonic runs, executions spanning more than one run: none


In [8]:
requests = {c: list(f.sort_values("turn")["tool_names"]) for c, f in turns.groupby("case")}
names = sorted(requests)

def fuse(seq, rule):
    """One synthetic run from recorded executions: (segment() result, true split)."""
    fn, labels, truth = shapes.RULES[rule], [], []
    for case in seq:
        start = len(labels)
        labels += [fn(tools) for tools in requests[case]]
        truth.append(list(range(start, len(labels))))
    return shapes.segment(labels), truth, labels

def boundaries(split):
    return {s[0] for s in split} - {0}

results = []
for rule in ["d09", "shapes7", "shapes9"]:
    pairs = list(itertools.permutations(names, 2))
    exact = sum(fuse(p, rule)[0] == fuse(p, rule)[1] for p in pairs)
    results.append({"rule": rule, "runs": "all pairs", "n": len(pairs),
                    "runs split exactly": exact / len(pairs), "boundaries found": exact / len(pairs)})
    rng = random.Random(28)
    for k in (3, 5, 8):
        n, exact, found, total = 2000, 0, 0, 0
        for _ in range(n):
            seg, truth, _ = fuse([rng.choice(names) for _ in range(k)], rule)
            exact += seg == truth
            found += len(boundaries(seg) & boundaries(truth))
            total += len(boundaries(truth))
        results.append({"rule": rule, "runs": f"{k} requests", "n": n,
                        "runs split exactly": exact / n, "boundaries found": found / total})
recovery = pd.DataFrame(results)
for col in ("runs split exactly", "boundaries found"):
    recovery[col] = recovery[col].map("{:.1%}".format)
recovery

,rule,runs,n,runs split exactly,boundaries found
0,d09,all pairs,812,63.8%,63.8%
1,d09,3 requests,2000,41.1%,62.7%
2,d09,5 requests,2000,15.7%,60.9%
3,d09,8 requests,2000,3.9%,61.6%
4,shapes7,all pairs,812,83.5%,83.5%
5,shapes7,3 requests,2000,65.0%,80.5%
6,shapes7,5 requests,2000,41.9%,79.8%
7,shapes7,8 requests,2000,24.1%,81.1%
8,shapes9,all pairs,812,84.7%,84.7%
9,shapes9,3 requests,2000,67.2%,81.7%


### What goes wrong, and what it costs

A missed boundary could come from anywhere, so the failures are grouped by the
shapes of the two requests that met. A merge only matters if it mislabels
something, so every recovered request is also checked for a wrong label, and
the shape shares per run are compared with the truth.

In [9]:
shape_of = {c: shapes.interaction_label([shapes.shapes7(t) for t in requests[c]]) for c in names}

failed = collections.Counter()
for a, b in itertools.permutations(names, 2):
    seg, truth, _ = fuse((a, b), "shapes7")
    if seg != truth:
        failed[(shape_of[a], shape_of[b])] += 1
misses = pd.DataFrame([{"first": a, "second": b, "pairs missed": n} for (a, b), n in failed.most_common()])
print(f"shapes7: {sum(failed.values())} of 812 pairs missed; "
      f"pairs of two different shapes among them: {sum(n for (a, b), n in failed.items() if a != b)}")
misses

shapes7: 134 of 812 pairs missed; pairs of two different shapes among them: 0


,first,second,pairs missed
0,factual.climate,factual.climate,42
1,venue.search,venue.search,42
2,factual.reference,factual.reference,30
3,scheduling.edit,scheduling.edit,12
4,scheduling.draft,scheduling.draft,6
5,factual.forecast,factual.forecast,2


In [10]:
rng = random.Random(28)
segments = mislabelled = 0
tv = []
for _ in range(2000):
    seq = [rng.choice(names) for _ in range(5)]
    seg, _, labels = fuse(seq, "shapes7")
    got = collections.Counter(shapes.interaction_label([labels[i] for i in s]) for s in seg)
    true = collections.Counter(shape_of[c] for c in seq)
    segments += len(seg)
    mislabelled += sum(1 for s in seg
                       if shapes.interaction_label([labels[i] for i in s]) not in {shape_of[c] for c in seq})
    g, t = sum(got.values()), sum(true.values())
    tv.append(0.5 * sum(abs(got[k] / g - true[k] / t) for k in set(got) | set(true)))
print(f"requests recovered: {segments}; carrying a shape absent from their run: {mislabelled}")
print(f"mean distance between recovered and true shape shares per run (total variation): {sum(tv) / len(tv):.3f}")

requests recovered: 8388; carrying a shape absent from their run: 0
mean distance between recovered and true shape shares per run (total variation): 0.086


### Reading the simulation

- **Rule C is deterministic, and it fails in only one way.** Every missed
  boundary is two requests of the same shape meeting, such as seafood then
  Sintra lunch. Requests of different shapes are always separated.
- **A merge never mislabels.** No recovered request carries a shape that was
  not in its run, and none becomes `other`. What a merge distorts is the
  *count* per shape, not which shapes appear.
- **A finer label finds more boundaries**, because fewer neighbours share a
  shape: `d09` finds about 62 %, `shapes7` about 80 %, `shapes9` about 82 %.
  The step from 4 to 7 classes is worth it; the step from 7 to 9 barely is,
  and costs the singleton classes seen in check 3.
- **Whole runs degrade with length.** The chance that a long run contains
  *some* same-shape neighbour grows with every request. So counting requests
  per session is not something rule C supports.

## 9. What this simulation cannot show

Every recorded case has exactly one turn that calls a tool. Fused runs can
therefore only produce the *merge* error, never the *split* error. A
participant's "find seafood" → "add the second one" is one request to them,
and rule C splits it into `venue.search` then `scheduling.edit`. Nothing
recorded looks like that, so its frequency is unknown until participant data
exists.

Random draws also ignore how people actually sequence requests. Real
sessions probably have *more* same-shape neighbours (a search, then "and
coffee?"). Whether that follow-up is a new request or a refinement is
ambiguous to a human reader too, which is one argument that a shape-level
analysis should not depend on the answer.

The fusion harness is itself reusable: it scores any segmenter against known
boundaries, including a model-based one, if Q29 goes that way.

## 10. Where the decisions stand

| # | question | recommendation | status |
|---|---|---|---|
| Q28 | which rule labels a turn's shape | `shapes7`, `other` as its own row; `advisory` = "called no tool"; `shapes9` kept as a secondary view | open, author |
| Q29 | how turns are grouped into requests | rule C for shape distribution and coverage (P1); turns stay the unit for failure rates (P2), thumbs and `user_action`; coverage reported per turn **and** per segment; no claim rests on a request count | open, author |

Both rules are versioned (`REQUEST_SHAPE_RULE_VERSION`, currently
`0-candidates`) and are frozen at C05, before any participant data exists.

**Decision trail (2026-09-23).** The author proposed taking the shapes the 29
cases identify and checking that choice against a full-suite `n = 1` run →
run `42cab17a` was used instead of a new run → the checks confirmed seven
shapes and exposed the advisory mismatch (§4) → the author noted that the
participant round has no request boundary → rule C was proposed → the author
asked whether C separates requests cleanly, testing it by fusing recorded
runs rather than trusting it → this simulation (§8). Having the agent decide
the boundary was raised as the fallback, and rejected on rule 4 (§7).